# VOICE_CLONE_TTS — Nhân bản giọng nói (Text-to-Speech) — Bản nguồn mở
## Cách dùng nhanh: bấm "Chạy tất cả" (Run all), đợi khoảng 5 phút cài đặt là dùng được.

VUI LÒNG ĐỌC HẾT HƯỚNG DẪN BÊN DƯỚI TRƯỚC KHI SỬ DỤNG.

---

## GIỚI THIỆU
* Bản xây dựng sạch từ mã nguồn mở **F5-TTS** (https://github.com/SWivid/F5-TTS), minh bạch, không dùng app mã hóa.
* Chức năng: **nhân bản giọng nói** — tải lên 1 file giọng mẫu (5–12 giây), nhập văn bản bất kỳ, máy sẽ đọc bằng đúng giọng đó.
* Hỗ trợ **KHÔNG GIỚI HẠN KÍ TỰ** (tự động chia nhỏ văn bản dài).
* **Đã tối ưu chống thiếu chữ / vấp**: văn bản được chia theo câu với độ dài cố định, mỗi đoạn được kết thúc đầy đủ dấu câu trước khi ghép nối.
* Model mặc định: **F5-TTS v1 Base (Anh + Trung Quốc)**. Trong giao diện có sẵn lựa chọn **model tiếng Việt**.

## 👉 HƯỚNG DẪN:
1. Trên trình đơn Colab: **Runtime -> Change runtime type -> T4 GPU** (cực kì quan trọng).
2. **Bấm "Chạy tất cả"** để cài đặt và khởi động.
3. Lần chạy đầu tiên sẽ tải model (~1.3 GB) và mất vài phút.
4. Sau khi có link giao diện (dòng cuối cùng của TASK 6), mở trong tab mới.
5. Tải lên **file giọng mẫu** (WAV/MP3 sạch, không nhạc nền, 5–12 giây), nhập văn bản, bấm **Synthesize**.
6. Muốn đọc **tiếng Việt**: trong giao diện chọn model `Tiếng Việt (toandev/F5-TTS-Vietnamese)`.

## NẾU VẪN CÒN THIẾU CHỮ / VẤP:
* Giảm thanh trượt **"Độ dài đoạn (max_chars)"** xuống 60–80.
* Nhập chính xác **"Nội dung trong file giọng mẫu"** (không để trống) để máy tính đúng độ dài.
* Dùng file giọng mẫu ngắn, rõ (5–10 giây), không nhạc nền.
* Tăng **nfe_step** lên 48–64 nếu bị rè hoặc vấp.

## ⚠️ LƯU Ý:
* Chỉ dùng cho mục đích cá nhân / học tập. **Không** nhân bản giọng nói của người khác khi chưa được phép.
* Model F5-TTS dùng giấy phép **CC-BY-NC** (phi thương mại).


In [ ]:
# @title TASK 1 — KIỂM TRA MÔI TRƯỜNG (GPU / FFMPEG)
import subprocess, sys

# 1) Kiểm tra GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CẢNH BÁO: Không có GPU. Vào Runtime -> Change runtime type -> chọn T4 GPU rồi chạy lại.")

# 2) Cài ffmpeg nếu chưa có (cần để đọc/xuất âm thanh)
try:
    subprocess.run(["ffmpeg", "-version"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    print("ffmpeg: OK")
except Exception:
    print("ffmpeg: đang cài đặt...")
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=False)
    print("ffmpeg: xong")


In [ ]:
# @title TASK 2 — CÀI ĐẶT F5-TTS
# Trên Colab, torch/torchaudio đã có sẵn. pip install f5-tts sẽ bổ sung các thư viện còn thiếu.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "f5-tts"], check=True)
print("Đã cài xong F5-TTS.")


In [ ]:
# @title TASK 3 — TẢI & NẠP MODEL (mặc định F5-TTS v1 Base)
import os
from f5_tts.api import F5TTS

VIETNAMESE_MODEL = False  # <--- đổi thành True nếu muốn model tiếng Việt làm mặc định

ENGINES = {}  # cache các engine đã nạp (dùng chung cho các task sau)

def load_engine(model_name="F5TTS_v1_Base", ckpt_file="", vocab_file=""):
    key = (model_name, ckpt_file)
    if key not in ENGINES:
        print(f"Đang tải và nạp model: {model_name} ...")
        ENGINES[key] = F5TTS(model=model_name, ckpt_file=ckpt_file, vocab_file=vocab_file)
    return ENGINES[key]

if VIETNAMESE_MODEL:
    from huggingface_hub import hf_hub_download
    ckpt = hf_hub_download("toandev/F5-TTS-Vietnamese", "model_latest.safetensors")
    vocab = hf_hub_download("toandev/F5-TTS-Vietnamese", "vocab.txt")
    engine = load_engine("F5TTS_Base", ckpt, vocab)
else:
    engine = load_engine("F5TTS_v1_Base")

print("OK — model đã sẵn sàng.")


In [ ]:
# @title TASK 4 — HÀM SINH ỔN ĐỊNH (chống thiếu chữ / vấp)
# Lý do gốc của lỗi "thiếu chữ" và "vấp": F5-TTS chia văn bản thành các đoạn
# (chunk) có độ dài tự động có thể rất lớn, và đoạn cuối mỗi chunk hay bị cắt mất từ.
# Hàm dưới đây ép chia theo CÂU với độ dài CỐ ĐỊNH, và thêm dấu câu đầy đủ cho mỗi đoạn.
import os, random, sys
import numpy as np
import torch
import torchaudio
import soundfile as sf
from f5_tts.infer.utils_infer import (
    preprocess_ref_audio_text,
    chunk_text,
    infer_batch_process,
    target_sample_rate,
    target_rms,
    cross_fade_duration,
    remove_silence_for_generated_wav,
)
from f5_tts.model.utils import seed_everything


def _split_clean(text, max_chars):
    """Chia văn bản theo câu, giới hạn mỗi đoạn ~max_chars ký tự, luôn kết thúc bằng dấu câu."""
    batches = chunk_text(text, max_chars=int(max_chars))
    out = []
    for b in batches:
        b = b.strip()
        if not b:
            continue
        if b[-1] not in ".!?。！？":
            b = b + "."
        out.append(b)
    return out


def stable_generate(engine, ref_file, ref_text, gen_text, max_chars=100,
                    speed=1.0, nfe=32, cfg=2.0, seed=None):
    """Sinh giọng ổn định: trả về (wave, sr, spec, seed_da_dung)."""
    if seed is None:
        seed = random.randint(0, sys.maxsize)
    seed_everything(seed)

    ref_audio, ref_text = preprocess_ref_audio_text(ref_file, ref_text, show_info=print)
    batches = _split_clean(gen_text, max_chars=max_chars)
    print("Số đoạn cần sinh:", len(batches))
    if not batches:
        return None, target_sample_rate, None, seed

    audio, sr = torchaudio.load(ref_audio)
    gen = infer_batch_process(
        (audio, sr),
        ref_text,
        batches,
        engine.ema_model,
        engine.vocoder,
        mel_spec_type=engine.mel_spec_type,
        progress=None,
        target_rms=target_rms,
        cross_fade_duration=cross_fade_duration,
        nfe_step=int(nfe),
        cfg_strength=float(cfg),
        sway_sampling_coef=-1.0,
        speed=speed,
        fix_duration=None,
        device=engine.device,
    )
    try:
        wave, s, spec = next(gen)
    except StopIteration:
        wave, s, spec = None, target_sample_rate, None
    return wave, s, spec, seed


def save_wav(wave, sr, path, remove_silence=False):
    sf.write(path, wave, sr)
    if remove_silence:
        remove_silence_for_generated_wav(path)
    return path


In [ ]:
# @title TASK 5 — CHẠY THỬ (dùng giọng mẫu có sẵn)
import os
from importlib.resources import files

os.makedirs("/content/output", exist_ok=True)

ref_audio = str(files("f5_tts").joinpath("infer/examples/basic/basic_ref_en.wav"))
ref_text = "Some call me nature, others call me mother nature."
gen_text = "Hello! This is a quick test of your cloned voice. Keep this file as your reference voice sample. " \
           "The text is split into small complete sentences so that every single word is pronounced fully and clearly."

wave, sr, spec, seed = stable_generate(engine, ref_audio, ref_text, gen_text,
                                       max_chars=100, speed=1.0, nfe=32, cfg=2.0)
out_path = save_wav(wave, sr, "/content/output/test_output.wav")
print("Đã tạo:", out_path, "| seed =", seed, "| sr =", sr)

from IPython.display import Audio
Audio(out_path)


In [ ]:
# @title TASK 6 — KHỞI ĐỘNG GIAO DIỆN NHÂN BẢN GIỌNG NÓI
import os
import gradio as gr
from huggingface_hub import hf_hub_download

MODELS = {
    "F5TTS_v1_Base (Anh + Trung)": ("F5TTS_v1_Base", "", ""),
    "Tiếng Việt (toandev/F5-TTS-Vietnamese)": ("F5TTS_Base", "toandev/F5-TTS-Vietnamese", "model_latest.safetensors"),
    "F5TTS_Base (Anh + Trung)": ("F5TTS_Base", "", ""),
}

os.makedirs("/content/output", exist_ok=True)

def synthesize(model_key, ref_audio, ref_text, gen_text, speed, nfe, cfg, max_chars, remove_silence, seed_text):
    if ref_audio is None:
        raise gr.Error("Vui lòng tải lên file giọng mẫu (ref audio).")
    if not (gen_text or "").strip():
        raise gr.Error("Vui lòng nhập văn bản cần đọc.")
    model_name, repo_id, ckpt_name = MODELS[model_key]
    key = model_name
    if key not in ENGINES:
        print("Đang tải model:", model_key)
        if repo_id:
            ckpt = hf_hub_download(repo_id, ckpt_name)
            vocab = hf_hub_download(repo_id, "vocab.txt")
            ENGINES[key] = F5TTS(model=model_name, ckpt_file=ckpt, vocab_file=vocab)
        else:
            ENGINES[key] = F5TTS(model=model_name)
    eng = ENGINES[key]
    seed = None if not (seed_text or "").strip() else int(seed_text)
    wave, sr, spec, seed_used = stable_generate(
        eng, ref_audio, (ref_text or ""), gen_text,
        max_chars=max_chars, speed=speed, nfe=nfe, cfg=cfg, seed=seed,
    )
    if wave is None:
        raise gr.Error("Không sinh được âm thanh. Kiểm tra lại file giọng mẫu / văn bản.")
    out = save_wav(wave, sr, "/content/output/cloned_output.wav", remove_silence=remove_silence)
    return out, f"Xong. seed = {seed_used} | {len(gen_text)} ký tự"

with gr.Blocks(title="VOICE_CLONE_TTS") as demo:
    gr.Markdown("### Nhân bản giọng nói (F5-TTS) — đã tối ưu chống thiếu chữ / vấp")
    with gr.Row():
        with gr.Column():
            model_key = gr.Dropdown(list(MODELS.keys()), value=list(MODELS.keys())[0], label="Model")
            ref_audio = gr.Audio(type="filepath", label="File giọng mẫu (5-12 giây, sạch)")
            ref_text = gr.Textbox(label="Nội dung trong file giọng mẫu (để trống = tự nhận dạng)", placeholder="Không bắt buộc, nhưng nhập chính xác thì đỡ thiếu chữ hơn")
            gen_text = gr.Textbox(label="Văn bản cần đọc (không giới hạn ký tự)", lines=6)
            with gr.Accordion("Tùy chọn nâng cao", open=False):
                max_chars = gr.Slider(40, 250, value=100, step=5, label="Độ dài đoạn (max_chars) — càng nhỏ càng ít thiếu chữ nhưng dễ vấp hơn")
                speed = gr.Slider(0.3, 2.0, value=1.0, step=0.05, label="Tốc độ đọc")
                nfe = gr.Slider(8, 64, value=32, step=1, label="Số bước khử nhiễu (nfe_step)")
                cfg = gr.Slider(1.0, 4.0, value=2.0, step=0.1, label="Cường độ bám giọng (cfg_strength)")
                seed_text = gr.Textbox(label="Seed (để trống = ngẫu nhiên)", placeholder="Không bắt buộc")
                remove_silence = gr.Checkbox(value=True, label="Loại bỏ khoảng lặng thừa")
        with gr.Column():
            out_audio = gr.Audio(type="filepath", label="Kết quả")
            info = gr.Textbox(label="Trạng thái", interactive=False)
    btn = gr.Button("Synthesize")
    btn.click(synthesize, inputs=[model_key, ref_audio, ref_text, gen_text, speed, nfe, cfg, max_chars, remove_silence, seed_text], outputs=[out_audio, info])

try:
    demo.launch(share=True, debug=False)
except Exception as e:
    print("Không tạo được link share, chuyển sang link local:", e)
    demo.launch(share=False, debug=False)

print("Dùng xong có thể đóng tab này lại. Muốn mở lại chỉ cần chạy lại cell này.")


In [ ]:
# @title DÙNG TRỰC TIẾP BẰNG HÀM (tùy chọn)
def clone_voice(ref_file, gen_text, ref_text="", output="/content/output/result.wav",
                speed=1.0, seed=None, max_chars=100, nfe=32, cfg=2.0):
    """Nhân bản giọng ổn định: ref_file = file giọng mẫu, gen_text = văn bản cần đọc."""
    eng = load_engine("F5TTS_v1_Base")
    wave, sr, spec, seed_used = stable_generate(eng, ref_file, ref_text, gen_text,
                                                max_chars=max_chars, speed=speed,
                                                nfe=nfe, cfg=cfg, seed=seed)
    path = save_wav(wave, sr, output, remove_silence=True)
    return path, sr, seed_used

# Ví dụ cách dùng (bỏ comment để chạy):
# out, sr, seed_used = clone_voice("/content/my_voice.wav", "Xin chào, đây là giọng nói được nhân bản.")
# from IPython.display import Audio
# Audio(out)


## CÁCH TẢI KẾT QUẢ VỀ MÁY
* Trong giao diện, bấm biểu tượng tải xuống (download) của ô audio kết quả.
* Hoặc vào thư mục `/content/output/` trong Files (panel bên trái) và tải xuống.

## MẸO CHẤT LƯỢNG
* File giọng mẫu nên dài **5–12 giây**, rõ ràng, không nhạc nền, không ồn.
* **Nhập chính xác "Nội dung trong file giọng mẫu"** — nếu để trống máy phải tự nhận dạng bằng Whisper (chậm hơn và có thể sai, gây thiếu chữ).
* Bị **thiếu chữ**: giảm "Độ dài đoạn (max_chars)".
* Bị **vấp / lắp**: tăng nfe_step lên 48–64, hoặc tăng nhẹ max_chars lên 120–150.
* Với văn bản rất dài, máy tự chia nhỏ và ghép nối — cứ nhập thoải mái.

## LIÊN HỆ MÃ NGUỒN
* F5-TTS: https://github.com/SWivid/F5-TTS
* Model tiếng Việt: https://huggingface.co/toandev/F5-TTS-Vietnamese
